In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, days, year, from_unixtime

print("⏳ Bước 0: Đang khởi động Siêu máy tính Single Node (32GB RAM - 8 Cores)...")
start_time = time.time()

# CẤU HÌNH SPARK MỚI: Bung xõa tài nguyên cho 1 máy duy nhất
spark = SparkSession.builder \
    .appName("AmazonReviews_Silver_SingleNode_Resume") \
    .config("spark.sql.catalog.gcp_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.gcp_catalog.type", "hadoop") \
    .config("spark.sql.catalog.gcp_catalog.warehouse", "gs://amazon-reviews-lakehouse-warehouse/warehouse/") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.executor.cores", "6") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.cleaner.referenceTracking.cleanTtl", "30") \
    .getOrCreate()

try:
    print("\n⏳ Bước 1: Quét hệ thống để tìm điểm tiếp sức...")
    df_bronze = spark.read.table("gcp_catalog.bronze.amazon_reviews")
    df_with_year = df_bronze.withColumn("review_year", year(from_unixtime(col("timestamp") / 1000)))
    df_with_year = df_with_year.filter((col("review_year") >= 1995) & (col("review_year") <= 2024))
    
    distinct_years = [row['review_year'] for row in df_with_year.select("review_year").distinct().orderBy("review_year").collect() if row['review_year'] is not None]
    
    # ⚠️ SỬA NĂM CẦN CHẠY TIẾP TẠI ĐÂY (Nếu máy cũ sập ở năm 2000, giữ nguyên số 2000)
    year_to_resume = 2019
    years_left_to_run = [y for y in distinct_years if y >= year_to_resume]
    
    print(f"📋 Các năm ĐÃ XONG (Sẽ bỏ qua): {[y for y in distinct_years if y < year_to_resume]}")
    print(f"🚀 Bắt đầu CHẠY TIẾP từ các năm: {years_left_to_run}")
    
    print("\n⏳ Bước 2 & 3: Kích hoạt chạy cuốn chiếu tốc độ cao...")
    window_spec = Window.partitionBy("asin", "user_id_hashed").orderBy(col("timestamp").desc())
    
    for i, current_year in enumerate(years_left_to_run):
        print(f"\n   🔄 Đang xử lý chunk: Năm {current_year} ({i+1}/{len(years_left_to_run)})...")
        
        df_chunk_bronze = df_with_year.filter(col("review_year") == current_year)
        df_chunk_clean = df_chunk_bronze.filter(col("rating").isNotNull() & col("text").isNotNull())
        df_chunk_silver = df_chunk_clean.withColumn("row_num", row_number().over(window_spec)) \
                                        .filter(col("row_num") == 1) \
                                        .drop("row_num", "review_year")
        
        writer = df_chunk_silver.writeTo("gcp_catalog.silver.amazon_reviews_cleaned") \
                                .tableProperty("write.format.default", "parquet") \
                                .tableProperty("write.parquet.compression-codec", "snappy")
        
        print(f"      Ghi nối tiếp (Append) dữ liệu {current_year} vào bảng Silver Iceberg...")
        writer.append()
            
        print(f"   ✅ Xong chunk năm {current_year}!")

    print("\n🎉 XUẤT SẮC! Toàn bộ 600 triệu dòng đã được xử lý xong trên cỗ máy mới!")
    
    print("\n⏳ Bước 4: Kiểm tra tổng thành quả Bậc Bạc...")
    total_silver_rows = spark.read.table("gcp_catalog.silver.amazon_reviews_cleaned").count()
    print(f"📊 Tổng số dòng SẠCH trong Data Lakehouse là: {total_silver_rows} dòng.")

except Exception as e:
    print(f"\n❌ LỖI TRONG QUÁ TRÌNH CHẠY:\n{str(e)}")

end_time = time.time()
print(f"\n⏱️ Tổng thời gian phiên làm việc hiện tại: {round(end_time - start_time, 2)} giây.")

⏳ Bước 0: Đang khởi động Siêu máy tính Single Node (32GB RAM - 8 Cores)...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 05:07:53 INFO SparkEnv: Registering MapOutputTracker
26/06/15 05:07:53 INFO SparkEnv: Registering BlockManagerMaster
26/06/15 05:07:53 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/15 05:07:53 INFO SparkEnv: Registering OutputCommitCoordinator



⏳ Bước 1: Quét hệ thống để tìm điểm tiếp sức...


📋 Các năm ĐÃ XONG (Sẽ bỏ qua): [1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
🚀 Bắt đầu CHẠY TIẾP từ các năm: [2019, 2020, 2021, 2022, 2023]

⏳ Bước 2 & 3: Kích hoạt chạy cuốn chiếu tốc độ cao...

   🔄 Đang xử lý chunk: Năm 2019 (1/5)...
      Ghi nối tiếp (Append) dữ liệu 2019 vào bảng Silver Iceberg...


   ✅ Xong chunk năm 2019!

   🔄 Đang xử lý chunk: Năm 2020 (2/5)...
      Ghi nối tiếp (Append) dữ liệu 2020 vào bảng Silver Iceberg...


   ✅ Xong chunk năm 2020!

   🔄 Đang xử lý chunk: Năm 2021 (3/5)...
      Ghi nối tiếp (Append) dữ liệu 2021 vào bảng Silver Iceberg...


   ✅ Xong chunk năm 2021!

   🔄 Đang xử lý chunk: Năm 2022 (4/5)...
      Ghi nối tiếp (Append) dữ liệu 2022 vào bảng Silver Iceberg...


   ✅ Xong chunk năm 2022!

   🔄 Đang xử lý chunk: Năm 2023 (5/5)...
      Ghi nối tiếp (Append) dữ liệu 2023 vào bảng Silver Iceberg...
